# 10 推理并行策略

## 简介

本笔记本介绍大语言模型推理阶段的并行策略。与训练不同，推理的显存瓶颈主要来自 **KV Cache**
（而非梯度和优化器状态），且计算模式在 Prefill 和 Decode 两阶段截然不同。

我们将讨论 KV Cache 分片、Prefill/Decode 阶段切换、以及推测解码（Speculative Decoding）等关键技术。

In [ ]:
import torch
import sys; sys.path.insert(0, '..')
from parallel.inference.kv_cache_shard import shard_kv_cache_by_heads, kv_cache_memory_analysis
from parallel.inference.prefill_decode import (
    analyze_prefill_characteristics,
    analyze_decode_characteristics,
    recommend_strategy,
)
from parallel.inference.speculative_decoding import speedup_analysis

print("推理并行模块加载完成")

## 1. KV Cache 分片

在自回归推理中，每生成一个 token 都需要访问之前所有 token 的 Key 和 Value
（即 KV Cache）。对于长序列和大 batch，KV Cache 的显存可能远超模型权重。

**按 Head 切分 KV Cache**:
这是最自然的做法——将注意力头分布到多张 GPU，每张 GPU 只存储部分头的 KV Cache。

```
  完整 KV Cache: (batch=1, n_heads=32, seq_len=8192, head_dim=128)
  切分到 4 GPU:
    GPU 0: heads[0:8]
    GPU 1: heads[8:16]
    GPU 2: heads[16:24]
    GPU 3: heads[24:32]
  
  显存节省: 约 75% / GPU (KV Cache 部分)
```

In [ ]:
# KV Cache 按 Head 切分演示
print("=" * 50)
print("KV Cache 按 Head 切分")
print("=" * 50)

batch, n_heads, seq_len, head_dim = 2, 32, 4096, 128
k = torch.randn(batch, n_heads, seq_len, head_dim)
v = torch.randn(batch, n_heads, seq_len, head_dim)

world_size = 4

print(f"完整 KV Cache 形状:")
print(f"  K: {list(k.shape)}")
print(f"  V: {list(v.shape)}")

for rank in range(world_size):
    k_local, v_local = shard_kv_cache_by_heads(k, v, rank, world_size)
    print(f"\nRank {rank} 本地 KV Cache:")
    print(f"  K: {list(k_local.shape)} (heads [{rank*8}:{(rank+1)*8}])")
    print(f"  V: {list(v_local.shape)}")

# 显存分析
print(f"\n" + "=" * 50)
print("KV Cache 显存分析")
print("=" * 50)

configs = [
    (1, 32, 1024, 128, "短对话 (1K tokens)"),
    (1, 32, 4096, 128, "中等对话 (4K)"),
    (8, 32, 8192, 128, "批量服务 (8 req, 8K)"),
    (1, 40, 32768, 128, "长上下文 (32K, DeepSeek-V3)"),
    (1, 64, 131072, 128, "超长上下文 (128K)"),
    (16, 40, 65536, 128, "高并发长上下文"),
]

print(f"\n{'场景':<30s} {'总KV(MB)':>10s} {'4GPU/卡(MB)':>12s} {'8GPU/卡(MB)':>12s}")
print("-" * 70)

for batch_sz, n_head, seq, h_dim, desc in configs:
    mem_4 = kv_cache_memory_analysis(batch_sz, n_head, seq, h_dim, n_gpus=4)
    mem_8 = kv_cache_memory_analysis(batch_sz, n_head, seq, h_dim, n_gpus=8)
    print(f"{desc:<30s} {mem_4['total_kv_cache_mb']:>10.1f} "
          f"{mem_4['per_gpu_sharded_mb']:>12.1f} {mem_8['per_gpu_sharded_mb']:>12.1f}")

print(f"\n★ 128K 上下文的 KV Cache 可能超过 10GB (fp16)")
print(f"★ 高并发场景下 KV Cache 分片是必需的")

## 2. Prefill vs Decode: 两阶段分析

大模型推理分为两个阶段，计算特征截然不同：

| 特征 | Prefill (预填充) | Decode (自回归解码) |
|---|---|---|
| **处理对象** | 完整 Prompt (S 个 token) | 每次 1 个新 token |
| **计算类型** | 大矩阵乘法 O(S×d²) | 小矩阵 + KV Cache 访问 |
| **瓶颈** | 计算密集型 (compute-bound) | 访存密集型 (memory-bound) |
| **推荐策略** | TP (张量并行) | DP 或 EP (数据/专家并行) |
| **GPU 利用率** | 高 (90%+) | 低 (10-30%) |

**核心洞察**: 最优并行策略应该在这两个阶段之间动态切换。
- Prefill 用 TP 加速大矩阵计算
- Decode 用 DP 并行服务多个请求，KV Cache 已就位

In [ ]:
# Prefill vs Decode 特征分析
print("=" * 50)
print("Prefill vs Decode 两阶段特征分析")
print("=" * 50)

# 不同序列长度下的分析
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
dim_model = 4096

print(f"\n模型维度: {dim_model}")
print(f"\n{'Seq Len':>8s} {'Phase':>8s} {'Compute Bound':>14s} {'Memory Bound':>14s} {'Strategy':>25s}")
print("-" * 80)

for s in seq_lengths:
    pref = analyze_prefill_characteristics(s, dim_model)
    dec = analyze_decode_characteristics(s, dim_model)
    print(f"{s:>8d}  {'Prefill':>8s} {str(pref['compute_bound']):>14s} "
          f"{'N/A':>14s} {pref['recommended_strategy']:>25s}")
    print(f"{'':>8s}  {'Decode':>8s} {'N/A':>14s} {str(dec['memory_bound']):>14s} "
          f"{dec['recommended_strategy']:>25s}")
    if s != seq_lengths[-1]:
        print()

# FLOPS 估算
print(f"\n" + "=" * 50)
print("计算量对比 (dim={dim_model}):")
print("=" * 50)
for s in seq_lengths[:4]:  # 只展示前几个
    pref = analyze_prefill_characteristics(s, dim_model)
    flops_b = pref['total_flops'] / 1e9
    print(f"  seq_len={s:5d}: Prefill FLOPS ≈ {pref['total_flops']/1e9:.2f}B, "
          f"Decode FLOPS ≈ 1 token × {dim_model ** 2 * 2 / 1e9:.2f}B")

print(f"\n★ Prefill 计算量随 seq_len 线性增长")
print(f"★ Decode 计算量恒定 (每次只处理 1 个 token)")
print(f"★ Decode 阶段 GPU 利用率低是推理吞吐的主要瓶颈")

In [ ]:
# 策略推荐
print("=" * 50)
print("Prefill → Decode 策略推荐")
print("=" * 50)

scenarios = [
    (512, 4096, 4, "短 Prompt, 4 GPU"),
    (2048, 4096, 8, "中等 Prompt, 8 GPU"),
    (8192, 4096, 8, "长 Prompt, 8 GPU"),
    (16384, 5120, 16, "长 Prompt, 16 GPU"),
    (32768, 8192, 32, "超长 Prompt, 32 GPU"),
]

print(f"\n{'场景':<25s} {'推荐策略'}")
print("-" * 65)
for seq_len, dim, n_gpus, desc in scenarios:
    strategy = recommend_strategy(seq_len, dim, n_gpus)
    print(f"{desc:<25s} {strategy}")

print(f"\n★ 长序列 (>4K) 建议 Prefill 用 TP 加速，Decode 切换为 DP")
print(f"★ 短序列直接 DP 即可，无需策略切换")
print(f"★ KV Cache 在切换时已在各 GPU 上就位")

## 3. 推测解码（Speculative Decoding）

自回归解码的主要瓶颈是**串行依赖**：必须等上一个 token 生成后才能生成下一个。
推测解码用小模型（draft model）快速生成多个候选 token，再用大模型（target model）并行验证。

```
  传统自回归:  T → T → T → T → T  (5 步, 每步 1 token)
  推测解码:    D → D → D → D → D  (draft 快速生成 5 个候选)
               └─── T ─────────┘   (target 一次前向验证全部)
  结果:         接受 4/5 → 等效加速 4~/2 步 = 2x+
```

**加速条件**:
1. Draft 模型远小于 Target 模型 (参数量比 10:1 ~ 100:1)
2. Draft 模型接受率高 (通常 > 70%)
3. Target 模型单次前向远快于逐 token 生成

**加速比公式**:
$$Speedup = \frac{T_{target} \times n_{accepted}}{T_{draft} + T_{target}}$$

其中 $T_{target}$ = target 模型单次前向时间, $T_{draft}$ = draft 模型生成候选时间, $n_{accepted}$ = 接受的 token 数

In [ ]:
# 推测解码加速比分析
print("=" * 50)
print("推测解码 (Speculative Decoding) 加速比分析")
print("=" * 50)

# 典型参数: Draft 模型 ~100M, Target 模型 ~7B
# Draft 单 token 生成: ~2ms, Target 单次前向: ~30ms

n_candidates = 5  # 候选 token 数

print(f"\n场景设定:")
print(f"  Draft 模型: ~100M 参数, 每 token ~2ms")
print(f"  Target 模型: ~7B 参数, 单次前向 ~30ms")
print(f"  候选 token 数: {n_candidates}")

# 不同接受率的加速比
print(f"\n不同接受率下的加速比:")
print(f"{'接受率':>10s} {'接受数':>8s} {'Draft耗时':>10s} {'Target耗时':>12s} {'总耗时':>10s} {'加速比':>8s}")
print("-" * 65)

for accepted in range(1, n_candidates + 1):
    draft_time = 10.0  # 5 tokens × 2ms
    target_time = 30.0  # 单次 target forward
    speedup = speedup_analysis(draft_time, target_time, accepted, n_candidates)
    acc_rate = accepted / n_candidates
    total_time = draft_time + target_time
    baseline_time = target_time * accepted
    print(f"{acc_rate:>9.0%} {accepted:>8d} {draft_time:>9.1f}ms {target_time:>11.1f}ms "
          f"{total_time:>9.1f}ms {speedup:>7.2f}x")

print(f"\n★ 接受率越高，加速越明显")
print(f"★ 接受率=100% 时理论 5x 加速 → 实际约 {speedup_analysis(10, 30, 5, 5):.1f}x")
print(f"★ 接受率=60% 时仍有正加速 → 坏情况最多浪费 1 次 target forward")

# 不同 Draft/Target 速度比下的加速效果
print(f"\n" + "=" * 50)
print("不同 Draft/Target 速度比:")
print("=" * 50)

ratios = [
    (2, 50, "极快 Draft vs 慢 Target (小模型+大模型)"),
    (5, 50, "快 Draft vs 慢 Target"),
    (10, 50, "中等 Draft vs 慢 Target"),
    (10, 20, "中等 Draft vs 中等 Target"),
    (20, 30, "慢 Draft vs 中等 Target (不利场景)"),
]

print(f"\n{'场景':<45s} {'Draft':>6s} {'Target':>7s} {'加速比(5/5)':>12s} {'加速比(3/5)':>12s}")
print("-" * 85)
for draft_t, target_t, desc in ratios:
    sp_full = speedup_analysis(draft_t, target_t, 5, 5)
    sp_partial = speedup_analysis(draft_t, target_t, 3, 5)
    print(f"{desc:<45s} {draft_t:>5.0f}ms {target_t:>6.0f}ms "
          f"{sp_full:>11.2f}x {sp_partial:>11.2f}x")

## 5. 相关文档

本 notebook 对应的详细原理文档：

- [推理并行详解](../../docs/parallel/inference.md) — KV Cache 分片、Prefill/Decode 分离、投机解码的完整原理
- [LLaMA 3 架构详解](../../docs/models/llama3.md) — KV Cache 的基础概念
- [通信原语详解](../../docs/parallel/communication.md) — KV Cache 分片中的通信模式

建议阅读顺序：先运行本 notebook 建立直觉，再阅读文档理解推理优化的工程细节。

## 5. 练习与思考

1. **KV Cache 显存**: 计算 LLaMA3-70B (n_heads=64, head_dim=128, n_layers=80) 在 seq_len=128K 时的 KV Cache 总大小
2. **投机解码加速比**: 分析 accept_rate 从 50% 到 95% 对加速比的影响曲线
3. **两阶段切换**: 设计一个 Prefill→Decode 动态切换策略，考虑 seq_len 和 batch_size

## 4. 推理并行策略总结

### 各策略在推理中的适用性

| 并行策略 | Prefill 阶段 | Decode 阶段 | 主要收益 |
|---|---|---|---|
| **TP (张量并行)** | 推荐 | 可选 | Prefill 加速大矩阵计算 |
| **PP (流水线并行)** | 可用 | 不推荐 | 超大模型跨节点推理 |
| **DP (数据并行)** | 不推荐 | 推荐 | 批量服务多请求 |
| **EP (专家并行)** | 可用 | 推荐 (MoE) | MoE 模型推理 |
| **KV Cache 分片** | N/A | 推荐 | 减少长序列/高并发显存 |
| **推测解码** | N/A | 推荐 | 加速自回归生成 |
| **CP (上下文并行)** | 推荐 (长序列) | 适用 | 超长上下文推理 |

### 关键要点

1. **KV Cache 是推理的主要显存瓶颈**，尤其长序列 + 高并发场景
2. **Prefill 计算密集** → 用 TP 加速；**Decode 访存密集** → 用 DP/EP
3. **推测解码** 用"投机取巧"打破串行依赖，接受率高时可获 2-5x 加速
4. **动态策略切换**：Prefill 用 TP，Decode 用 DP，是现代推理服务的关键优化
5. **推理与训练的并行策略设计哲学不同**：训练追求吞吐，推理追求延迟 + 吞吐的平衡